# Phase 1: Data Understanding & Schema Inspection

**Milestone**: Milestone 1.5 (Raw Data Ingestion Module)  
**Dataset**: CPCB Surface Water Quality Monitoring Dataset (`data/interim/cpcb_labeled.csv`)  
**Objective**: Ingest, validate, and explore the labeled CPCB dataset using the centralized `src.data.load_data` module.

In [ ]:
# Cell 1: Environment Setup & Module Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.load_data import load_raw_data, validate_schema, load_config

print("Loaded src.data.load_data successfully.")

In [ ]:
# Cell 2: Central Data Ingestion & Frozen Schema Validation
config = load_config()
df = load_raw_data(config=config, validate=True)

print(f"\n[Schema Validation Success]")
print(f"  * Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  * Target Column: {config['target_column']}")
print(f"  * Candidate Features: {config['feature_columns']}")

## 1. Ground-Truth Target Distributions

Inspect the distributions for the primary multi-class target (`wqi_class`) and the secondary binary target (`Potability`).

In [ ]:
# Multi-class target breakdown
print("=== Primary Target (wqi_class) Breakdown ===")
class_counts = df["wqi_class"].value_counts()
class_pcts = df["wqi_class"].value_counts(normalize=True) * 100
target_dist = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)})
display(target_dist)

# Binary potability breakdown
print("\n=== Secondary Target (Potability) Breakdown ===")
pot_counts = df["Potability"].value_counts()
pot_pcts = df["Potability"].value_counts(normalize=True) * 100
pot_dist = pd.DataFrame({"Count": pot_counts, "Percentage (%)": pot_pcts.round(2)})
pot_dist.index = ["Non-Potable (0)", "Potable (1)"]
display(pot_dist)

## 2. Frozen Candidate Features Audit

Inspect descriptive statistics and missingness for the four candidate water quality parameters.

In [ ]:
features = config["feature_columns"]
stats_df = df[features].describe().T
stats_df["missing_count"] = df[features].isna().sum()
stats_df["missing_pct"] = (df[features].isna().mean() * 100).round(2)
display(stats_df[["count", "mean", "std", "min", "50%", "max", "missing_count", "missing_pct"]])

## 3. Geographic & Administrative Scope

Analyze monitoring stations and state representation across Maharashtra and Uttar Pradesh.

In [ ]:
print(f"Total Unique Stations: {df['Station'].nunique()}")
print(f"Total Unique Districts: {df['District'].nunique()}")
print("\nState Representation:")
display(df.groupby("State").agg(Total_Observations=("Station", "count"), Unique_Stations=("Station", "nunique")))

## 4. Target Leakage Verification

Confirm that the continuous mathematical index `wqi_score` is strictly excluded from `feature_columns`.

In [ ]:
assert "wqi_score" not in config["feature_columns"], "CRITICAL: Target leakage in feature_columns!"
print("Target Leakage Verification: PASSED (wqi_score is excluded from features)")
print("\nData Understanding Notebook complete. Ready for Phase 2 Preprocessing & Feature Engineering.")